Write a query that will calculate the number of shipments per month. The unique key for one shipment is a combination of shipment_id and sub_id. Output the year_month in format YYYY-MM and the number of shipments in that month.

In [0]:
%sql
CREATE TABLE amazon_shipment (
  shipment_date DATE,
  shipment_id BIGINT,
  sub_id BIGINT,
  weight BIGINT
)

In [0]:
%sql
INSERT INTO amazon_shipment (
    shipment_id, sub_id, weight, shipment_date
)
VALUES
    (101, 1, 10, '2021-08-30'),
    (101, 2, 20, '2021-09-01'),
    (101, 3, 10, '2021-09-05'),
    (102, 1, 50, '2021-09-02'),
    (103, 1, 25, '2021-09-01'),
    (103, 2, 30, '2021-09-02'),
    (104, 1, 30, '2021-08-25'),
    (104, 2, 10, '2021-08-26'),
    (105, 1, 20, '2021-09-02');

In [0]:
%sql
select year_month, COUNT(shipment_id) as count from
(
select 
DATE_FORMAT(shipment_date, 'yyyy-MM') as year_month, shipment_id, sub_id
from amazon_shipment
) T
group by T.year_month;

In [0]:
import pandas as pd

df_amazon_shipment = spark.table("amazon_shipment").toPandas()

df_amazon_shipment['shipment_date'] = pd.to_datetime(df_amazon_shipment['shipment_date'])
df_amazon_shipment['year_month'] = df_amazon_shipment['shipment_date'].dt.strftime('%Y-%m')
df_amazon_shipment = df_amazon_shipment.groupby('year_month').agg({'shipment_date': 'count'}).reset_index()
df_amazon_shipment.columns = ['year_month', 'count']
display(df_amazon_shipment)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

df_amazon_shipment = spark.table("amazon_shipment")
df_amazon_shipment = df_amazon_shipment.withColumn('year_month',F.date_format(F.col('shipment_date'),'yyyy-MM'))
df_amazon_shipment = df_amazon_shipment.groupBy('year_month').agg(F.count('shipment_id').alias('count'))
display(df_amazon_shipment)
